In [2]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import datetime
# from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from schema import AnswerQuestion, ReflectAnswer
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq


In [3]:
load_dotenv()
# llm=llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

structured_llm_responder=llm.with_structured_output(AnswerQuestion)
structured_llm_revisor=llm.with_structured_output(ReflectAnswer)

d:\programming\LangChain impelentation\LangGraph\venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='250 words answer to the question'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


In [4]:
actor_prompt_template=ChatPromptTemplate.from_messages(
    [
        ("system",
         """You are an expert AI researcher.
         CUrrent time={time}
         
         1.{instruction}
         2.Reflect and critique on the above statement.
         3.After reflection and critiquing, list 1-3 search quaries to 
         research improvements, Do not include them inside the reflection""",),
         MessagesPlaceholder(variable_name='messages'),
         ("system","Answer the user question above in the required format"),
    ]
).partial(time=lambda: datetime.datetime.now().isoformat(),)

In [5]:
revisor_prompt="""Provide a 250 words detailed revised answer addressing all the critisisms mentioned in
the previous one and improving the answer with the contens from the search results.
-You must address the critisms mentioned in the missing and superfluous.
-The answer must contain citations, supporting all the information in the answer.
-The word limit should be strictly less that on equal to 250
"""

responder_prompt_template=actor_prompt_template.partial(instruction="Provide a 250 words detatiled answer")
revisor_prompt_template=actor_prompt_template.partial(instructions=revisor_prompt)

In [6]:
responder_chain=responder_prompt_template | structured_llm_responder
revisor_chain=revisor_prompt_template |structured_llm_revisor

In [7]:
response=responder_chain.invoke({"messages":[HumanMessage(
    content="Write a blog post on how startups can leverage AI to increase their growth rate")]})

In [8]:
print(response)

answer='Startups can harness AI to accelerate growth by leveraging data-driven decision-making, automation, and personalized customer experiences. First, AI-powered analytics tools enable startups to extract actionable insights from customer behavior, market trends, and operational metrics. For example, predictive analytics can forecast demand, optimize pricing strategies, and identify high-value leads, reducing guesswork in resource allocation. Second, automation streamlines repetitive tasks such as customer support (via chatbots), lead qualification, and content creation, freeing teams to focus on innovation. Tools like HubSpot or Zendesk integrate AI to enhance efficiency. Third, AI-driven personalization—such as tailored product recommendations or dynamic email campaigns—boosts customer retention and lifetime value. Platforms like Amazon Personalize or Adobe Target make this accessible even for small teams. Additionally, AI can optimize operational efficiency by automating supply c